# 리포트 플레이스홀더 초기화

Google Colab에서 실행합니다.  
`ml_leaderboard_representatives.json`에 등록된 ML 실험과 GNN 실험을 읽어  
대시보드 리포트 저장 경로에 빈 `report.json` 파일을 생성합니다.

**생성 위치** (`MyDrive/Graph_AML/dashboard/` 하위)
```
dashboard/
  GNN Result/{실험명}/report.json
  ML Result/{실험 레이블}/report.json
  Univariate Analysis/{실험 레이블}/report.json
```

이미 존재하는 파일은 건드리지 않습니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## CONFIG — 필요시 이 섹션만 수정하세요

In [2]:
from pathlib import Path

DRIVE_BASE_DIR = Path("/content/drive/MyDrive/Graph_AML")
ML_DIR         = DRIVE_BASE_DIR / "ml"
GNN_LOGS_DIR   = DRIVE_BASE_DIR / "gnn" / "logs"
DASHBOARD_DIR  = DRIVE_BASE_DIR / "dashboard"

## Engine — 수정하지 마세요

In [3]:
import json
import re

REPS_PATH = ML_DIR / "ml_leaderboard_representatives.json"

# ── 실험 레이블 헬퍼 (app.py 와 동일) ────────────────────────────────────
def _woe_iv_folder_name(ml_folder: str) -> str:
    m = re.match(r"(ml-\d+)", ml_folder)
    return m.group(1) if m else ml_folder

def _is_valid_rep(rep: dict) -> bool:
    folder = _woe_iv_folder_name(rep.get("ml_folder", ""))
    return bool(re.match(r"ml-\d+", folder)) and folder != "ml-00"

def _exp_label(rep: dict) -> str:
    note   = rep.get("note", "")
    folder = _woe_iv_folder_name(rep["ml_folder"])
    return f"{folder}  {'— ' + note if note else ''}".strip()

# ── ML 실험 목록 ─────────────────────────────────────────────────────────
if not REPS_PATH.exists():
    raise FileNotFoundError(f"representatives.json 없음: {REPS_PATH}")

reps      = json.loads(REPS_PATH.read_text(encoding="utf-8"))
ml_labels = [_exp_label(r) for r in reps if _is_valid_rep(r)]

# ── GNN 실험 목록 ─────────────────────────────────────────────────────────
gnn_exps = []
if GNN_LOGS_DIR.exists():
    gnn_exps = sorted(p.stem for p in GNN_LOGS_DIR.glob("*.log"))

# ── 탭 × 실험 매핑 ───────────────────────────────────────────────────────
TABS = {
    "GNN Result":          gnn_exps,
    "ML Result":           ml_labels,
    "Univariate Analysis": ml_labels,
}

total = sum(len(v) for v in TABS.values())
print(f"처리 대상: {total}개")
for tab, exps in TABS.items():
    print(f"  [{tab}] {len(exps)}개")
    for e in exps:
        print(f"    · {e}")

처리 대상: 7개
  [GNN Result] 3개
    · small_hi_epoch100_patience20_numneighs100100
    · small_hi_epoch100_patience20_numneighs5020
    · small_hi_epoch100_patience20_numneighs5020_tds_on
  [ML Result] 2개
    · ml-01  — ML-01, 파라미터 고정, 고카디널리티 컬럼(bank code, bank pair code)을 제외한 정책으로 학습
    · ml-02  — ML-02, ML-01과 동일 정책, optuna 하이퍼 파라미터 튜닝 적용함
  [Univariate Analysis] 2개
    · ml-01  — ML-01, 파라미터 고정, 고카디널리티 컬럼(bank code, bank pair code)을 제외한 정책으로 학습
    · ml-02  — ML-02, ML-01과 동일 정책, optuna 하이퍼 파라미터 튜닝 적용함


## 실행

In [4]:
created  = 0
skipped  = 0

for tab_name, exps in TABS.items():
    print(f"\n[{tab_name}]")
    for exp_name in exps:
        folder = DASHBOARD_DIR / tab_name / exp_name
        folder.mkdir(parents=True, exist_ok=True)
        report_path = folder / "report.json"
        if report_path.exists():
            print(f"  ✓ 이미 존재: {exp_name}")
            skipped += 1
        else:
            report_path.write_text("{}", encoding="utf-8")
            print(f"  + 생성됨:   {exp_name}")
            created += 1

print(f"\n완료 — 생성 {created}개 / 스킵 {skipped}개")


[GNN Result]
  + 생성됨:   small_hi_epoch100_patience20_numneighs100100
  + 생성됨:   small_hi_epoch100_patience20_numneighs5020
  + 생성됨:   small_hi_epoch100_patience20_numneighs5020_tds_on

[ML Result]
  + 생성됨:   ml-01  — ML-01, 파라미터 고정, 고카디널리티 컬럼(bank code, bank pair code)을 제외한 정책으로 학습
  + 생성됨:   ml-02  — ML-02, ML-01과 동일 정책, optuna 하이퍼 파라미터 튜닝 적용함

[Univariate Analysis]
  + 생성됨:   ml-01  — ML-01, 파라미터 고정, 고카디널리티 컬럼(bank code, bank pair code)을 제외한 정책으로 학습
  + 생성됨:   ml-02  — ML-02, ML-01과 동일 정책, optuna 하이퍼 파라미터 튜닝 적용함

완료 — 생성 7개 / 스킵 0개
